In [ ]:
# ============================================================================
# NOTEBOOK 04 — DESCARGA DE DEMANDA ELÉCTRICA DESDE LA API REData
# ============================================================================
# Objetivo: Descargar el histórico de demanda peninsular horaria de España
# desde la API pública apidatos.ree.es, año por año, y guardar los JSON
# crudos en /opt/spark-data/raw/redata/ para procesarlos después con PySpark.
# ============================================================================

import requests
import json
import os
import time
import calendar

RUTA_RAW_REDATA = "/opt/spark-data/raw/redata"
os.makedirs(RUTA_RAW_REDATA, exist_ok=True)

URL_BASE = "https://apidatos.ree.es/es/datos/demanda/evolucion"

print(f"Carpeta de destino: {RUTA_RAW_REDATA}")
print(f"Endpoint:           {URL_BASE}")

Carpeta de destino: /opt/spark-data/raw/redata
Endpoint:           https://apidatos.ree.es/es/datos/demanda/evolucion


In [11]:
def descargar_demanda_mes(anio: int, mes: int) -> dict:
    """
    Descarga la demanda peninsular horaria para un mes concreto.
    
    La API REData limita ventanas largas con granularidad horaria, por eso
    descargamos mes a mes en lugar de año completo.
    
    Parámetros:
        anio: año a descargar (ej. 2024)
        mes:  mes (1-12)
    
    Devuelve:
        dict con la respuesta JSON de la API.
    """
    ultimo_dia = calendar.monthrange(anio, mes)[1]
    
    params = {
        "start_date":  f"{anio}-{mes:02d}-01T00:00",
        "end_date":    f"{anio}-{mes:02d}-{ultimo_dia:02d}T23:59",
        "time_trunc":  "hour",             # granularidad horaria
        "geo_trunc":   "electric_system",  # tipo de agregación
        "geo_limit":   "peninsular",       # sistema peninsular
        "geo_ids":     8741,               # ID del sistema peninsular en REData
    }
    
    response = requests.get(URL_BASE, params=params, timeout=60)
    response.raise_for_status()
    
    return response.json()


# Test rápido sobre enero 2023 para verificar que todo funciona
print("Probando descarga de enero 2023...")
data_test = descargar_demanda_mes(2023, 1)
n_test = len(data_test['included'][0]['attributes']['values'])
print(f"  ✓ HTTP 200 - {n_test} puntos horarios recibidos")
print(f"  Primer punto: {data_test['included'][0]['attributes']['values'][0]}")

Probando descarga de enero 2023...
  ✓ HTTP 200 - 744 puntos horarios recibidos
  Primer punto: {'value': 19872.637, 'percentage': 1, 'datetime': '2023-01-01T00:00:00.000+01:00'}


In [ ]:
ANIO_INICIO = 2023
ANIO_FIN = 2026
MES_FIN = 4   # abril 2026 (último mes completo disponible)

MAX_REINTENTOS = 3
ESPERA_ENTRE_PETICIONES = 1  # segundos, cortesía con la API pública

# Generar lista de (año, mes) desde enero 2023 hasta abril 2026
periodos = []
for anio in range(ANIO_INICIO, ANIO_FIN + 1):
    for mes in range(1, 13):
        if anio == ANIO_FIN and mes > MES_FIN:
            break
        periodos.append((anio, mes))

print(f"Periodo: {ANIO_INICIO}-01 a {ANIO_FIN}-{MES_FIN:02d}")
print(f"Total de meses a descargar: {len(periodos)}")
print(f"Tiempo estimado: ~{len(periodos) * 2 / 60:.1f} minutos\n")

resumen = []

for anio, mes in periodos:
    ruta_salida = f"{RUTA_RAW_REDATA}/demanda_{anio}_{mes:02d}.json"
    
    if os.path.exists(ruta_salida):
        size_kb = os.path.getsize(ruta_salida) / 1024
        print(f"  {anio}-{mes:02d} → ya descargado ({size_kb:.1f} KB)")
        resumen.append((anio, mes, "skip", 0))
        continue
    
    # Intentamos descargar con reintentos
    exito = False
    for intento in range(1, MAX_REINTENTOS + 1):
        try:
            print(f"  {anio}-{mes:02d} → descargando...", end=" ")
            data_mes = descargar_demanda_mes(anio, mes)
            
            n_puntos = len(data_mes['included'][0]['attributes']['values'])
            
            with open(ruta_salida, 'w', encoding='utf-8') as f:
                json.dump(data_mes, f, ensure_ascii=False)
            
            size_kb = os.path.getsize(ruta_salida) / 1024
            print(f"OK ({n_puntos} puntos, {size_kb:.1f} KB)")
            resumen.append((anio, mes, "ok", n_puntos))
            exito = True
            break
        except Exception as e:
            print(f"FALLO ({type(e).__name__}: {str(e)[:60]})")
            if intento < MAX_REINTENTOS:
                time.sleep(5)
    
    if not exito:
        resumen.append((anio, mes, "error", 0))
    
    time.sleep(ESPERA_ENTRE_PETICIONES)

# Resumen final
print("\n" + "="*60)
print("RESUMEN DE DESCARGA")
print("="*60)
print(f"Total peticiones: {len(periodos)}")
print(f"  OK:    {sum(1 for _,_,e,_ in resumen if e=='ok')}")
print(f"  Skip:  {sum(1 for _,_,e,_ in resumen if e=='skip')}")
print(f"  Error: {sum(1 for _,_,e,_ in resumen if e=='error')}")
total_puntos = sum(p for _,_,e,p in resumen if e == 'ok')
print(f"\nPuntos descargados en esta ejecución: {total_puntos:,}")
print("="*60)

Periodo: 2023-01 a 2026-04
Total de meses a descargar: 40
Tiempo estimado: ~1.3 minutos

  2023-01 → descargando... OK (744 puntos, 61.5 KB)
  2023-02 → descargando... OK (672 puntos, 55.6 KB)
  2023-03 → descargando... OK (743 puntos, 61.3 KB)
  2023-04 → descargando... OK (720 puntos, 59.5 KB)
  2023-05 → descargando... OK (744 puntos, 61.4 KB)
  2023-06 → descargando... OK (720 puntos, 59.5 KB)
  2023-07 → descargando... OK (744 puntos, 61.4 KB)
  2023-08 → descargando... OK (744 puntos, 61.4 KB)
  2023-09 → descargando... OK (720 puntos, 59.5 KB)
  2023-10 → descargando... OK (745 puntos, 61.5 KB)
  2023-11 → descargando... OK (720 puntos, 59.5 KB)
  2023-12 → descargando... OK (744 puntos, 61.4 KB)
  2024-01 → descargando... OK (744 puntos, 61.4 KB)
  2024-02 → descargando... OK (696 puntos, 57.5 KB)
  2024-03 → descargando... OK (743 puntos, 61.3 KB)
  2024-04 → descargando... OK (720 puntos, 59.5 KB)
  2024-05 → descargando... OK (744 puntos, 61.4 KB)
  2024-06 → descargando... 

In [14]:
print("Ficheros descargados en raw/redata/:\n")

ficheros = sorted(os.listdir(RUTA_RAW_REDATA))
total_size_mb = 0
total_puntos = 0

for f in ficheros:
    ruta = os.path.join(RUTA_RAW_REDATA, f)
    size_kb = os.path.getsize(ruta) / 1024
    total_size_mb += size_kb / 1024
    
    # Contar puntos releyendo el JSON
    with open(ruta) as fp:
        data = json.load(fp)
    n_puntos = len(data['included'][0]['attributes']['values'])
    total_puntos += n_puntos
    
    print(f"  {f:30s}  {size_kb:>6.1f} KB   {n_puntos:>4} puntos")

print(f"\n{'TOTAL:':32s}  {total_size_mb*1024:>6.1f} KB   {total_puntos:>4} puntos")
print(f"\nTamaño total:    {total_size_mb:.2f} MB")
print(f"Total ficheros:  {len(ficheros)}")
print(f"Total registros: {total_puntos:,}")

Ficheros descargados en raw/redata/:

  demanda_2023_01.json              61.5 KB    744 puntos
  demanda_2023_02.json              55.6 KB    672 puntos
  demanda_2023_03.json              61.3 KB    743 puntos
  demanda_2023_04.json              59.5 KB    720 puntos
  demanda_2023_05.json              61.4 KB    744 puntos
  demanda_2023_06.json              59.5 KB    720 puntos
  demanda_2023_07.json              61.4 KB    744 puntos
  demanda_2023_08.json              61.4 KB    744 puntos
  demanda_2023_09.json              59.5 KB    720 puntos
  demanda_2023_10.json              61.5 KB    745 puntos
  demanda_2023_11.json              59.5 KB    720 puntos
  demanda_2023_12.json              61.4 KB    744 puntos
  demanda_2024_01.json              61.4 KB    744 puntos
  demanda_2024_02.json              57.5 KB    696 puntos
  demanda_2024_03.json              61.3 KB    743 puntos
  demanda_2024_04.json              59.5 KB    720 puntos
  demanda_2024_05.json            

In [16]:
print("="*60)
print("RESUMEN DESCARGA REData")
print("="*60)
print(f"Endpoint:        {URL_BASE}")
print(f"Geografía:       Sistema peninsular (geo_ids=8741)")
print(f"Granularidad:    Horaria (time_trunc=hour)")
print(f"Cobertura:       {ANIO_INICIO}-01 → {ANIO_FIN}-{MES_FIN:02d}")
print(f"Ficheros JSON:   {len(ficheros)}")
print(f"Tamaño total:    {total_size_mb:.2f} MB")
print(f"Total registros: {total_puntos:,}")
print(f"Ruta destino:    {RUTA_RAW_REDATA}")
print("="*60)

RESUMEN DESCARGA REData
Endpoint:        https://apidatos.ree.es/es/datos/demanda/evolucion
Geografía:       Sistema peninsular (geo_ids=8741)
Granularidad:    Horaria (time_trunc=hour)
Cobertura:       2023-01 → 2026-04
Ficheros JSON:   40
Tamaño total:    2.35 MB
Total registros: 29,183
Ruta destino:    /opt/spark-data/raw/redata
